In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import os


from tueplots import bundles, cycler, figsizes
from tueplots.constants.color import palettes
import tueplots.constants.color.palettes as tuepalt

In [ ]:
def set_style(rows=0, cols=0, column="full", dpi=100):
    # plt.rcParams.update(bundles.icml2024())
    plt.rcParams.update(cycler.cycler(color=palettes.tue_plot))
    plt.rcParams.update({"figure.dpi": dpi})
    if (rows > 0) and (cols > 0):
        plt.rcParams.update(bundles.icml2024(column=column, nrows=rows, ncols=cols))

In [ ]:

relevant_emotions = ["angry", "neutral", "sad"]

In [ ]:
os.makedirs("fig", exist_ok=True)

In [ ]:
path = "../data/newspaper_collection_evaluation_results_20_12_2025.csv"
np_coll_df = pd.read_csv(path, index_col=False)

In [ ]:
# some preprocessing
np_coll_df["date"] = pd.to_datetime(np_coll_df["date"])
emotions_str = ["angry","fear", "happy", "sad", "surprise", "neutral"]

In [ ]:
np_coll_df = np_coll_df.query("party != 'fdp'")
np_coll_df = np_coll_df.drop(columns=["disgust"])

In [ ]:
np_coll_df = np_coll_df.query("party != 'transnational'")

In [ ]:
min_confidence = 70
cut_off_confidence_df = np_coll_df.query("confidence > @min_confidence")
confidence_mean = cut_off_confidence_df["confidence"].mean()
print("*================================================*")
print(f"| {len(cut_off_confidence_df)}/{len(np_coll_df)} Samples remain with confidence > {min_confidence} |")
print(f"|       Resulting confidence mean: {confidence_mean:.2f}         |")
print("*================================================*")
np_coll_df = cut_off_confidence_df

In [ ]:
print(f"#Parties={len(np_coll_df["party"].unique())}\n#Newspaper={len(np_coll_df["newspaper"].unique())}\n#Emotions={len(np_coll_df["dominant_emotion"].unique())}")

In [ ]:
from scipy.stats import chi2_contingency
def chi2_pairwise_newspapers(df, newspaper_A, newspaper_B, target_emotion, target_party):
    subset = df.query("party == @target_party").copy()
    
    subset = subset[subset['newspaper'].isin([newspaper_A, newspaper_B])]
    subset['is_emotion'] = subset['dominant_emotion'] == target_emotion

    count_A_target = subset[(subset['newspaper'] == newspaper_A) & (subset['is_emotion'])].shape[0]
    count_A_other = subset[(subset['newspaper'] == newspaper_A) & (~subset['is_emotion'])].shape[0]
    
    count_B_target = subset[(subset['newspaper'] == newspaper_B) & (subset['is_emotion'])].shape[0]
    count_B_other = subset[(subset['newspaper'] == newspaper_B) & (~subset['is_emotion'])].shape[0]
    
    if (count_A_target + count_A_other == 0) or (count_B_target + count_B_other == 0):
        print(f"cA {newspaper_A}:", count_A_target + count_A_other)
        print(f"cB {newspaper_B}:", count_B_target + count_B_other)
        return None

    obs = [[count_A_target, count_A_other], [count_B_target, count_B_other]]
    chi2, p_val, _, _ = chi2_contingency(obs)
    
    prop_A = count_A_target / (count_A_target + count_A_other)
    prop_B = count_B_target / (count_B_target + count_B_other)
    
    return {
        "newspaper_A": newspaper_A,
        "newspaper_B": newspaper_B,
        "party": target_party,
        "emotion": target_emotion,
        "n_faces_A": count_A_target + count_A_other,
        "n_faces_B": count_B_target + count_B_other,
        "prop_A": round(prop_A, 3),
        "prop_B": round(prop_B, 3),
        "diff": prop_A - prop_B,
        "p_value": p_val,
        "significant": p_val < 0.05
    }

In [ ]:
all_newspapers = np_coll_df["newspaper"].unique().tolist()
all_test_results = []
# for each newspaper...
print(all_newspapers)
for i, npA in enumerate(all_newspapers):
    # pick a emotion...
    for target_emotion in relevant_emotions:
        # and a party
        for target_party in np_coll_df["party"].unique().tolist():
            test_results = []
            # and compare the distribution of (party, newspaper, emotion)
            # against every different newspaper
            for npB in all_newspapers[:]:
                if npB == npA: continue # don't compare against itself
                
                # compute p-value and other statistics
                res = chi2_pairwise_newspapers(
                    np_coll_df,
                    newspaper_A=npA,
                    newspaper_B=npB,
                    target_party=target_party,
                    target_emotion=target_emotion
                )
                
                if res: # add result, if there is something
                    test_results.append(res)
                else:
                    print(f"No results for comparison of {npA} vs. {npB}")
            
            if not test_results:
                continue

            all_test_results.append(pd.DataFrame(test_results)) 

all_test_results_df = pd.concat(all_test_results)
# filer out symmetric pairs
all_test_results_df = all_test_results_df.query("diff >= 0")

In [ ]:
len(all_test_results_df)

In [ ]:
from statsmodels.stats.multitest import multipletests

rejected, p_adjusted, _, _ = multipletests(
    all_test_results_df["p_value"], 
    alpha=0.05, 
    method="fdr_bh"
)

all_test_results_df["p_value_corrected"] = p_adjusted
all_test_results_df["significant_corrected"] = rejected

In [ ]:
all_test_results_df

In [ ]:
significant_abs = len(all_test_results_df.query("significant_corrected == True"))
significant_proportion =  significant_abs / len(all_test_results_df)

print(f"Total significant results: {significant_abs}/{len(all_test_results_df)} ({significant_proportion*100:.1f}%)")

In [ ]:

set_style(1, 3)
plt.rcParams.update(figsizes.icml2024_full())
fig, ax = plt.subplots(1, 3)

for i, emo in enumerate(relevant_emotions):
    sns.scatterplot(
        data=all_test_results_df.query("emotion == @emo"),
        x="newspaper_A",
        y="p_value_corrected",
        hue="newspaper_A",
        ax=ax[i],
        legend=True,
        alpha=0.5,
        style="newspaper_B"
    )
    ax[i].set_ylabel("p-value ({})".format(emo))
    ax[i].axhline(0.05, color='red', linestyle='--')

handles, labels = ax[0].get_legend_handles_labels()
for ax_ in ax:
    ax_.get_legend().remove()
fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.2),
    ncol=7,
    frameon=False,
)

plt.savefig("fig/kde_test.pdf", bbox_inches='tight')
plt.show()

In [ ]:
r, c = 1, 3
set_style(r, c, column="full")
fig, ax = plt.subplots(r, c)
p_val_upper_thresh = .05
for i, emo in enumerate(relevant_emotions):
    sns.stripplot(
        data=all_test_results_df.query("(emotion == @emo) and (p_value_corrected < @p_val_upper_thresh)"),
        y="newspaper_A",
        x="p_value",
        hue="party",
        ax=ax[i],
        legend=True,
        alpha=1,
        size=7,
    )
    ax[i].set_ylabel("p-value ({})".format(emo))
    ax[i].axvline(0.05, color='red', linestyle='--', linewidth=0.5)
    ax[i].set_xlim(-0.00001, p_val_upper_thresh)

handles, labels = ax[0].get_legend_handles_labels()
"""
for ax_ in ax:
    ax_.get_legend().remove()
fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.1),
    ncol=7,
    frameon=False
)
"""
plt.savefig("fig/kde_test.pdf", bbox_inches='tight')
plt.show()

In [ ]:
r, c = 1, 3
set_style(r, c, column="full")
fig, ax = plt.subplots(r, c)
p_val_upper_thresh = .1

newspaper_Bs = all_test_results_df["newspaper_B"].unique()
newspaper_As = all_test_results_df["newspaper_A"].unique()
unique_parties = all_test_results_df["party"].unique()

marker_map = {"stern": "*", "freitag": "X", "nd": "D", "spiegel": "s", "taz": "P", "compact": "h", "sz": ">"}
party_palette = {p: palettes.tue_plot[i] for i, p in enumerate(unique_parties)}

for i, emo in enumerate(relevant_emotions):
    base_query = "(emotion == @emo) and (p_value < @p_val_upper_thresh)"
    
    for np_B, marker_sym in marker_map.items():
        subset = all_test_results_df.query(f"{base_query} and (newspaper_B == @np_B) and p_value_corrected <= 0.05")
        if subset.empty:
            continue
        np_A_order = [np_A for np_A in newspaper_As if len(subset.query("newspaper_A == @np_A")) > 0]
        sns.swarmplot(
            data=subset,
            y="newspaper_A",
            x="p_value_corrected",
            hue="party",
            hue_order=unique_parties,
            order=np_A_order,
            palette=party_palette,
            ax=ax[i],
            marker=marker_sym,
            alpha=1,
            size=10,
            legend=False
        )
    
    ax[i].set_ylabel("party ({})".format(emo))
    ax[i].set_xlabel("p-value")
    ax[i].axvline(0.05, color='red', linestyle='--', linewidth=0.5)
    ax[i].set_xlim(-0.001, p_val_upper_thresh)

party_handles = [
    plt.Line2D([0], [0], marker='o', color=party_palette[p], markersize=5, label=p)
    for p in unique_parties
]

newspaper_handles = [
    plt.Line2D([0], [0], marker=marker_map[n], color='w', markerfacecolor='k', markersize=5, label=n)
    for n in newspaper_Bs
]

fig.legend(
    handles=party_handles + newspaper_handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.2),
    ncol=7,
    frameon=False
)

plt.savefig("fig/emotion_newspaper_party_chi2_test_results.png", bbox_inches='tight')
plt.show()

In [ ]:
all_test_results_df.query(
    "p_value_corrected < 0.05 and newspaper_A == 'sz'"
)

In [ ]:
significant_results = all_test_results_df.query("significant_corrected == True").sort_values(by="diff", ascending=False)
significant_results.head(5)

In [ ]:
np_bias_counts = {"newspaper": [], "count": []}
for np_ in all_test_results_df["newspaper_A"].unique():
    np_bias_counts["newspaper"].append(np_)
    np_bias_counts["count"].append(len(significant_results.query(
        "newspaper_A == @np_ or newspaper_B == @np_"
    )))
np_bias_counts_df = pd.DataFrame(np_bias_counts).sort_values(by="count", ascending=False)

In [ ]:
set_style(1, 1, "half")
fig, ax = plt.subplots()
sns.scatterplot(
    data=np_bias_counts_df,
    x="newspaper",
    hue="newspaper",
    y="count",
    ax=ax,
    legend=False,
    zorder=4
)
sns.lineplot(
    data=np_bias_counts_df,
    x="newspaper",
    y="count",
    ax=ax,
    zorder=2,
    legend=False
)
plt.show()
# plt.savefig("fig/count_test.png")

... did not observe significance for **Taz**!

In [ ]:
set_style(1, 1, "half")
fig, ax = plt.subplots()
# emotion_styles = {"sad": "X", "angry": "x"}
sns.scatterplot(
    data=significant_results,
    x="newspaper_B", y="newspaper_A",
    size="diff",
    hue="party",
    style="emotion",
    sizes=(100, 200),
    ax=ax
)
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0)
# ax.margins(x=0.5, y=0.5)
plt.savefig("fig/diffs.png")

In [ ]:
def capitalize_first_letter(string):
    first_letter = string[0].upper()
    res = string[1:]
    result = first_letter + res
    return result

In [ ]:
all_test_results_df

In [ ]:
significant_results["newspaper_pair"] = significant_results["newspaper_A"].apply(lambda x: capitalize_first_letter(x)) \
                                        + " - " \
                                        + significant_results["newspaper_B"].apply(lambda x: capitalize_first_letter(x))
set_style(1, 1, "half", dpi=300)
fig, ax = plt.subplots()

# Custom color palette for parties
party_palette = {
    "union": palettes.tue_plot[1],     # gray
    "linke": palettes.tue_plot[0],     # red
    "gruenen": palettes.tue_plot[5]    # green
}

sns.scatterplot(
    data=significant_results.sort_values("newspaper_pair"),
    x="diff",
    y="newspaper_pair",
    hue="party",
    palette=party_palette,
    style="emotion",
    markers={emotion: 'v' if emotion == "sad" else 's' if emotion == "neutral" else 'X' for emotion in significant_results["emotion"].unique()},
    marker='o',
    s=90,
    alpha=0.9,
    ax=ax,
    zorder=4
)

ax.grid(axis="y", linestyle="--", alpha=0.5, zorder=4)
leg = ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0, markerscale=0.75)

# Customize legend labels
label_map = {
    "union": "Union",
    "linke": "Linke",
    "gruenen": "Grünen",
    "sad": "Sad",
    "neutral": "Neutral",
    "angry": "Angry",
    "emotion": "Emotion",
    "party": "Party"
}
for text in leg.get_texts():
    current_text = text.get_text()
    if current_text in label_map:
        text.set_text(label_map[current_text])
    # Change font size for legend section titles
    if current_text in ['party', 'emotion']:
        text.set_fontsize(8)  # Adjust size

ax.set_xlabel("Difference between proportional occurence of emotions")
ax.set_ylabel("")
plt.savefig("fig/newspaper_bias_results_cleveland_dot_plot.pdf")

## Why is there so much union?
- **Compact**, **Freitag** and **Nd** display **Union** more **Sad**  than **Spiegel**, **Stern** and **Sz**

On the other hand 

- **Spiegel**, **Stern** and **Sz**, display the **Union** more angry than **Freitag**, **Nd**, and (**Compact**)  
- And Sz also even more often than Spiegel

In [ ]:
import numpy as np
def get_emotion_party_np_politicians(emotion, party, newspaper):
    # subset the corresponding party and newspaper entries
    subset = np_coll_df.query(
        "party == @party and newspaper == @newspaper"
    ).copy()

    # prepare to count emotion occurence
    subset["is_emotion"] = subset["dominant_emotion"] == emotion
    # number of total entries for the party in the the specific newspaper
    N = len(subset)

    # count how often policticians have the displayed emotion 
    politicians = subset["surname"].unique()
    counts = np.array([subset.query("surname == @n")["is_emotion"].sum() for n in politicians])
    # proportion of an politicans being display with the @emotion out of ALL entries of the newspaper/party 
    proportionals = counts / N

    return pd.DataFrame({"name": politicians, "count": counts, "proportion": proportionals}).sort_values(by="count", ascending=False)

In [ ]:
def show_politician_emotion_distr(emotion, newspaper,party, ax):
    df = get_emotion_party_np_politicians(emotion, party, newspaper)
    sns.barplot(
        data=df,
        x="name",
        hue="name",
        y="proportion",
        ax=ax
    )
    ax.set_title(f"{emotion} (prop:{df["proportion"].sum():.2f}) in {newspaper}")
    ax.set_ylabel(f"% of total emotions")
    ax.set_xticks(ax.get_xticks())
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")


In [ ]:
r, c = 2, 3
set_style(r, c, "full", dpi=200)
fig, ax = plt.subplots(r, c)
for i, npr in enumerate(["compact", "freitag", "nd"]):
    show_politician_emotion_distr("sad", npr, "union", ax=ax[0, i])
for i, npr in enumerate(["sz", "spiegel", "stern"]):
    show_politician_emotion_distr("sad", npr, "union", ax=ax[1, i])

plt.show()

Merkel is just overall very sad.. :(

In [ ]:
r, c = 2, 3
set_style(r, c, "full", dpi=200)
fig, ax = plt.subplots(r, c)
for i, npr in enumerate(["compact", "freitag", "nd"]):
    show_politician_emotion_distr("angry", npr, "union", ax=ax[0, i])
for i, npr in enumerate(["sz", "spiegel", "stern"]):
    show_politician_emotion_distr("angry", npr, "union", ax=ax[1, i])

plt.show()

so in the outlets that display the union less angry **angela** has the largest proportion of _angry_, wheres in the others friedrich is the main contributor

In [ ]:
merkel_df = np_coll_df.query("surname == 'merkel'")

In [ ]:
def n_emotions_np(newspaper, party):
    len_np = len(np_coll_df.query("newspaper == @newspaper and party == @party"))
    return len_np

def get_proportion_politician_newspaper(politician, newspaper, party):
    len_np = n_emotions_np(newspaper, party)
    n_pol = len(np_coll_df.query("surname == @politician and newspaper == @newspaper"))
    return n_pol / len_np

def plot_politicians_proportions(politicians, newspaper, party, ax):
    y = [get_proportion_politician_newspaper(p, newspaper, party) for p in politicians]
    print("PROPS:", y)
    sns.barplot(
        x=politicians,
        y=y,
        hue=politicians,
        ax=ax
    )
    ax.set_title(f"{newspaper}")

In [ ]:
r, c = 2, 3
set_style(r, c, "full", dpi=200)
fig, ax = plt.subplots(r, c)

politicians = ["merz", "merkel"]
for i, npr in enumerate(["compact", "freitag", "nd"]):
    plot_politicians_proportions(politicians, npr, "union", ax=ax[0, i])
for i, npr in enumerate(["sz", "spiegel", "stern"]):
    plot_politicians_proportions(politicians, npr, "union", ax=ax[1, i])

fig.suptitle("Proportions of Merkel/Merz occuring in the Newspapers")
plt.show()

Maybe Merz has more angry images $\Rightarrow$ showing him more often leads to more angry pictures.

In [ ]:
def compare_merz_merkel(newspaper):
    merz_angry = len(np_coll_df.query(
        "surname == 'merz' and dominant_emotion == 'angry' and newspaper == @newspaper"
    ))
    merz_total = len(np_coll_df.query("surname == 'merz' and newspaper == @newspaper"))
    merkel_angry = len(np_coll_df.query(
        "surname == 'merkel' and dominant_emotion == 'angry' and newspaper == @newspaper "
    ))
    merkel_total = len(np_coll_df.query("surname == 'merkel' and newspaper == @newspaper"))
    return merz_angry / merz_total, merkel_angry / merkel_total

In [ ]:
r, c = 2, 3
set_style(r, c, "full", dpi=200)
fig, ax = plt.subplots(r, c)

politicians = ["merz", "merkel"]
for i, npr in enumerate(["compact", "freitag", "nd"]):
    merz_angry, merkel_angry = compare_merz_merkel(npr)
    sns.barplot(
        x=["merkel", "merz"],
        y=[merkel_angry, merz_angry],
        hue=["merkel", "merz"],
        ax=ax[0, i]
    )
    ax[0, i].set_title(f"{npr}")
for i, npr in enumerate(["sz", "spiegel", "stern"]):
    merz_angry, merkel_angry = compare_merz_merkel(npr)
    sns.barplot(
        x=["merkel", "merz"],
        y=[merkel_angry, merz_angry],
        hue=["merkel", "merz"],
        ax=ax[1, i]
    )
    ax[1, i].set_title(f"{npr}")
fig.suptitle("Percentage of dom. emotion being angry per person") 
plt.show()

This shows that merz is 2-3 times more often displayed angry.
Reasons could be that images are picked consciously (intended way of protraying), that merz is generally more angry (maybe gender bias?). But this is not really inferable from our data 

In [ ]:
melted_df = np_coll_df.melt(
    id_vars=["surname", "newspaper"],
    value_vars=["sad", "angry"],
    var_name="emotion",
    value_name="prop" 
).query("surname in ['merz', 'merkel', 'spahn']")

In [ ]:
r, c = 1, 1
set_style(r, c, dpi=300, column="half")
fig, ax = plt.subplots()
plot_data = (
    melted_df.query("surname in ['merkel', 'merz', 'spahn']")
    .sort_values("prop")
    .assign(
        ecdf=lambda df: (df.groupby(["emotion", "surname"]).cumcount() + 1) 
        / df.groupby(["emotion", "surname"])["prop"].transform("count")
    )
)

sns.lineplot(
    data=plot_data,
    x="prop",
    y="ecdf",
    hue="emotion",
    style="surname",
    ax=ax,
    drawstyle="steps-post"
)
ax.set_xlabel("Proportion of emotion detected in images ")

In [ ]:
r, c = 2, 3
set_style(r, c, "full", dpi=300)
fig, ax = plt.subplots(r, c)
emotions = plot_data["emotion"].unique()
surnames = plot_data["surname"].unique()


politicians = ["merz", "merkel"]
for i, npr in enumerate(["compact", "freitag", "nd"]):
    sns.lineplot(
        data=plot_data.query("newspaper == @npr"),
        x="prop",
        y="ecdf",
        hue="emotion",
        style="surname",
        hue_order=emotions,
        style_order=surnames,
        ax=ax[0, i],
        drawstyle="steps-post"
)
    ax[0, i].set_xlabel("Proportion of emotion detected in images ")
    ax[0, i].legend(fontsize=5)
for i, npr in enumerate(["sz", "spiegel", "stern"]):
    sns.lineplot(
        data=plot_data.query("newspaper == @npr"),
        x="prop",
        y="ecdf",
        hue="emotion",
        style="surname",
        hue_order=emotions,
        style_order=surnames,
        ax=ax[1, i],
        drawstyle="steps-post"
)
    ax[1, i].legend(fontsize=5)

In [ ]:
emotions = plot_data["emotion"].unique()
surnames = plot_data["surname"].unique()

group_1 = ["compact", "freitag", "nd"]
group_2 = ["spiegel", "stern", "sz"]

r, c = 1, 2
set_style(r, c, "half", dpi=300)
fig, ax = plt.subplots(r, c)

sns.lineplot(
    data=plot_data.query("newspaper in @group_1"),
    x="prop",
    y="ecdf",
    hue="surname",
    style="emotion",
    hue_order=surnames,
    style_order=emotions,
    ax=ax[0],
    drawstyle="steps-post",
    legend=False,
    linewidth=0.8
)
ax[0].set_xlabel("Prop. of emotion (G1)")

sns.lineplot(
    data=plot_data.query("newspaper in @group_2"),
    x="prop",
    y="ecdf",
    hue="surname",
    style="emotion",
    hue_order=surnames,
    style_order=emotions,
    ax=ax[1],
    drawstyle="steps-post",
    linewidth=0.8
)
ax[1].set_ylabel("")
ax[1].set_xlabel("Prop. of emotion (G2)")

# Create custom legend with markers for surnames
handles, labels = ax[1].get_legend_handles_labels()
# Extract the actual colors from the legend handles
from matplotlib.lines import Line2D
surname_colors = {}
for handle, label in zip(handles, labels):
    if label in surnames:
        surname_colors[label] = handle.get_color()

# Create custom handles with markers
custom_handles = []
for label in labels:
    if label in surnames:
        custom_handles.append(Line2D([0], [0], marker='o', color='w', 
                                    markerfacecolor=surname_colors.get(label, 'gray'), 
                                    markersize=6, label=label))
    else:
        # Keep emotion style lines as they are
        idx = labels.index(label)
        custom_handles.append(handles[idx])
# Customize legend labels

label_map = {
    "merz": "Merz",
    "spahn": "Spahn",
    "merkel": "Merkel",
    "surname": "Surname",
    "sad": "Sad",
    "angry": "Angry",
    "emotion": "Emotion"
}

leg = ax[1].legend(handles=custom_handles, labels=labels, bbox_to_anchor=(1.05, 1), 
            fontsize=5, loc="upper left", borderaxespad=0)

# Change font size for legend section titles
for text in leg.get_texts():
    if text.get_text() in ['surname', 'emotion']:
        text.set_fontsize(8)  # Adjust size
    if text.get_text() in label_map:
        text.set_text(label_map[text.get_text()])

# move dotted lines under solid lines
for a in ax:
    for line in a.get_lines():
        if line.get_linestyle() in [':', '--', 'dotted', 'dashed']:
            line.set_zorder(5)
        else:
            line.set_zorder(1)
plt.savefig("fig/emotion_dst_g1_g2.pdf")

In [ ]:
for p in significant_results["party"].unique():
    print(f"{p}: {len(significant_results.query("party == @p"))}")

## Some numbers for the paper

In [ ]:
print("Mean =", np_bias_counts_df["count"].mean())
print("Std = ", np_bias_counts_df["count"].std())

In [ ]:
print(significant_results.query("newspaper_A in @group_1 and emotion == 'sad'")["diff"].mean())

In [ ]:
print(significant_results.query("newspaper_A in @group_2 and emotion == 'angry'")["diff"].mean())

### How often are Merkzel and Merz portrait in Group 1 and 2?

In [ ]:
g1_df = np_coll_df.query("newspaper in @group_1 and party == 'union'")
ng1_df = len(g1_df)
g2_df = np_coll_df.query("newspaper in @group_2 and party == 'union'")
ng2_df = len(g2_df)

print("="*10, "Abs", "="*10)
merkel_g1 = len(g1_df.query("surname == 'merkel'"))
merkel_g2 = len(g2_df.query("surname == 'merkel'"))
merz_g1 = len(g1_df.query("surname == 'merz'"))
merz_g2 = len(g2_df.query("surname == 'merz'"))
spahn_g1 = len(g1_df.query("surname == 'spahn'"))
spahn_g2 = len(g2_df.query("surname == 'spahn'"))

print("Abs G1:", ng1_df)
print("Abs G2:", ng2_df)
print("Merkel G1:", merkel_g1)
print("Merkel G2:", merkel_g2)
print("Merz G1:", merz_g1)
print("Merz G2:", merz_g2)
print("="*10, "Proportions", "="*10)
print(f"Merkel G1: {merkel_g1 / ng1_df:.2f}")
print(f"Merkel G2: {merkel_g2 / ng2_df:.2f}")
print(f"Merz G1: {merz_g1 / ng1_df:.2f}")
print(f"Merz G2: {merz_g2 / ng2_df:.2f}")
print(f"Spahn G1: {spahn_g1 / ng1_df:.2f}")
print(f"Spahn G2: {spahn_g2 / ng2_df:.2f}")


In [ ]:
len(np_coll_df.query(
    "newspaper in @group_1 and surname == 'merz'"
))

In [ ]:
for p in np_coll_df.query("party == 'union'")["surname"].unique():
    print(f"G1: {p}: {len(g1_df.query("surname == @p"))/ng1_df}")
    print(f"G2: {p}: {len(g2_df.query("surname == @p"))/ng2_df}")
    print("=")

In [ ]:
print(g1_df.query("surname == 'spahn'")[["angry", "sad"]].mean())
print(g2_df.query("surname == 'spahn'")[["angry", "sad"]].mean())